# 1.3 在真实数据集上训练 BPE

本节在 TinyStories 和 OpenWebText 两个数据集上独立训练 BPE，观察真实语料得到的词表与合并规则。

In [6]:
import os
import sys

# 直接指定项目根目录，后面的路径一律相对它来写；需替换成自己实际的项目根目录的绝对路径
os.chdir(r"C:\Users\intangible\Desktop\easy-llm")

# 注意切目录并不会让 import 找到 scripts/ 下的模块，还得把它加进模块搜索路径
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))

print("工作目录:", os.getcwd())
# 工作目录: C:\Users\intangible\Desktop\easy-llm

工作目录: C:\Users\intangible\Desktop\easy-llm


In [3]:
# 实际训练在项目根目录执行；输入文本需要提前准备好。
# TinyStories
# train_bpe("data/TinyStories/TinyStoriesV2-GPT4-train.txt", 10000, ["<|endoftext|>"])

# OpenWebText
# train_bpe("data/openwebtext/owt_train.txt", 32000, ["<|endoftext|>"])

In [7]:
# 训练完毕，载入训练好的分词器查看结果
import json
from pathlib import Path
from bpe_tokenizer import Tokenizer

def load_tokenizer(dataset_name):
    base = Path("results/tokenizer") / dataset_name
    with open(base / "vocab.json", encoding="utf-8") as f:
        raw_vocab = json.load(f)
        
    # 以词表中保存的特殊 token 为准，兼容已有实验产物。
    special_token = bytes.fromhex(raw_vocab["256"]).decode("utf-8")
    tokenizer = Tokenizer()
    tokenizer.from_files(base / "vocab.json", base / "merge.txt", [special_token])
    return tokenizer, special_token

tokenizers = {
    "TinyStories": load_tokenizer("TinyStoriesV2-GPT4-train"),
    "OpenWebText": load_tokenizer("owt_train"),
}
for name, (tokenizer, special_token) in tokenizers.items():
    print(name, "词表大小:", len(tokenizer.vocab), "合并规则条数:", len(tokenizer.merges))

TinyStories 词表大小: 10000 合并规则条数: 9743
OpenWebText 词表大小: 32000 合并规则条数: 31743


In [8]:
def longest_bpe_token(tokenizer, special_token):
    special_bytes = special_token.encode("utf-8")
    candidates = {
        i: token for i, token in tokenizer.vocab.items()
        if i >= 256 and token != special_bytes
    }
    longest_id = max(candidates, key=lambda i: len(candidates[i]))
    return longest_id, candidates[longest_id]

for name, (tokenizer, special_token) in tokenizers.items():
    token_id, token_bytes = longest_bpe_token(tokenizer, special_token)
    
    # TinyStories 干净，最长 token 是正常的英文；
    # OpenWebText 混有被重复编码的脏数据，会得到 ÃÂÃÂ 这种看似乱码的超长 token；
    # 实际大模型训练管线中，原始数据集肯定是要经过一系列数据预处理步骤（去重，去乱码，修复编码等等）后再送入分词器和模型中进行训练的
    print(name, "最长 BPE token:", token_id, repr(token_bytes))
    print("  文本:", token_bytes.decode("utf-8", errors="replace"))

TinyStories 最长 BPE token: 7162 b' accomplishment'
  文本:  accomplishment
OpenWebText 最长 BPE token: 25822 b'\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82\xc3\x83\xc3\x82'
  文本: ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ
